# Duplicates Before Splitting — the leak nobody notices

01 Core Python · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · **▶ 07 Model Prep** · 08 Case Studies

`07_Model_Preparation/01_duplicate_detection_before_splitting.ipynb`

---

### In one paragraph (no jargon)

This looks like a repeat of the duplicates notebook in section 03, and the code is similar — but the reason is completely different and much more serious. If the same record appears twice and the split puts one copy in training and one in test, the model has *already seen the answer*. Your test score is then measuring memory, not learning, and it will be far too optimistic. **De-duplicate before you split, never after.** That ordering is the whole point of this notebook.

### After this notebook you can

- Explain why duplicates matter more before a split than anywhere else
- Detect exact and near-duplicates in a modelling dataset
- Resolve conflicts when two 'duplicate' rows disagree on a value
- Sequence the cleaning steps so no information leaks across the split


### What's inside

1. Setup and first look
2. Detecting duplicates
3. Resolving conflicting duplicates
4. Verifying the dataset is clean
5. ⚡ Why the order of operations matters
6. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Term | Plain English |
|---|---|
| **Training set** | The rows the model learns from. Usually 60–80%. |
| **Validation set** | Rows used to choose between models and tune settings. You look at these many times. |
| **Test set** | Rows kept sealed until the very end, used once, to get an honest performance estimate. |
| **Overfitting** | The model memorised the training rows instead of learning the pattern. Great on train, poor on anything new. |
| **Data leakage** | Information from the test set influenced training, so your score is flattering and meaningless. |
| **Stratified split** | Splitting so each part keeps the same class balance as the whole — essential when one outcome is rare. |
| **Cross-validation** | Rotating which slice is held out, so every row is tested on exactly once. More reliable than a single split. |
| **random_state** | A fixed seed making the "random" split reproducible. Always set it. |

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os
import seaborn as sns
sns.set_theme(style="whitegrid")

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def dataset_path(filename, rebuild=None):
    """Return a real path to `filename`, materialising a temp copy if it's missing.

    WHY: a few pandas tools (pd.ExcelFile, pd.read_sql) need an actual file path
         rather than a DataFrame, so the fallback has to be written to disk.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if rebuild is None:
        raise FileNotFoundError(filename)
    import tempfile
    tmp = os.path.join(tempfile.mkdtemp(prefix='bda_'), filename)
    frame = rebuild()
    (frame.to_excel(tmp, index=False) if filename.lower().endswith(('.xlsx', '.xls'))
     else frame.to_csv(tmp, index=False))
    print(f"'{filename}' not found -> wrote a rebuilt copy to {tmp}")
    return tmp

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    built = rebuild()
    if 'chunksize' in read_kwargs:            # keep chunked reads working on the fallback path
        size = read_kwargs['chunksize']
        return (built.iloc[i:i + size] for i in range(0, len(built), size))
    return built

def rebuild_student_data():
    """Exact copy of student_data.csv, embedded so this notebook never needs the file."""
    import io
    csv_text = """Student_ID,Student_Name,Age,Gender,State,City,Program,Year_of_Study,Residence_Type,Email,Phone_Number
STU001,Aarav Sharma,20,Male,Uttar Pradesh,Lucknow,BBA,2,Hostel,aarav.sharma@example.com,9876501001
STU002,Ananya Iyer,21,Female,Tamil Nadu,Chennai,BCom,3,Day Scholar,ananya.iyer@example.com,9876501002
STU003,Rohan Desai,19,Male,Maharashtra,Pune,BSc Data Science,1,Hostel,rohan.desai@example.com,9876501003
STU004,Meera Nair,22,Female,Kerala,Kochi,MBA,1,Hostel,meera.nair@example.com,9876501004
STU005,Arjun Reddy,20,Male,Telangana,Hyderabad,BTech,2,Day Scholar,arjun.reddy@example.com,9876501005
STU006,Sneha Patil,21,Female,Maharashtra,Nashik,BBA,3,Hostel,sneha.patil@example.com,9876501006
STU007,Vikram Singh,23,Male,Rajasthan,Jaipur,MBA,2,Hostel,vikram.singh@example.com,9876501007
STU008,Priya Das,19,Female,West Bengal,Kolkata,BCom,1,Day Scholar,priya.das@example.com,9876501008
STU009,Karthik Rao,20,Male,Karnataka,Bengaluru,BSc Data Science,2,Hostel,karthik.rao@example.com,9876501009
STU010,Neha Gupta,22,Female,Delhi,New Delhi,MBA,1,Day Scholar,neha.gupta@example.com,9876501010
STU011,Rahul Verma,21,Male,Madhya Pradesh,Indore,BBA,3,Hostel,rahul.verma@example.com,9876501011
STU012,Aditi Joshi,20,Female,Gujarat,Ahmedabad,BTech,2,Day Scholar,aditi.joshi@example.com,9876501012
STU013,Sanjay Kumar,19,Male,Bihar,Patna,BCom,1,Hostel,sanjay.kumar@example.com,9876501013
STU014,Pooja Kulkarni,21,Female,Maharashtra,Nagpur,BSc Data Science,3,Hostel,pooja.kulkarni@example.com,9876501014
STU015,Nikhil Menon,22,Male,Kerala,Thiruvananthapuram,MBA,2,Day Scholar,nikhil.menon@example.com,9876501015
STU016,Ishita Bose,20,Female,West Bengal,Siliguri,BBA,2,Hostel,ishita.bose@example.com,9876501016
STU017,Manish Yadav,23,Male,Haryana,Gurugram,MBA,2,Day Scholar,manish.yadav@example.com,9876501017
STU018,Divya Shetty,19,Female,Karnataka,Mangaluru,BCom,1,Hostel,divya.shetty@example.com,9876501018
STU019,Aditya Mishra,21,Male,Odisha,Bhubaneswar,BTech,3,Hostel,aditya.mishra@example.com,9876501019
STU020,Kavya Pillai,20,Female,Kerala,Kozhikode,BSc Data Science,2,Day Scholar,kavya.pillai@example.com,9876501020
STU021,Aarav Sharma,22,Male,Punjab,Ludhiana,MBA,1,Hostel,aarav.sharma2@example.com,9876501021
STU022,Tanvi Shah,19,Female,Gujarat,Surat,BBA,1,Day Scholar,tanvi.shah@example.com,9876501022
STU023,Mohit Saini,20,Male,Uttarakhand,Dehradun,BCom,2,Hostel,mohit.saini@example.com,9876501023
STU024,Ritika Chawla,21,Female,Punjab,Amritsar,BTech,3,Hostel,ritika.chawla@example.com,9876501024
STU025,Harsh Vardhan,22,Male,Jharkhand,Ranchi,MBA,1,Day Scholar,harsh.vardhan@example.com,9876501025
STU026,Simran Kaur,20,Female,Punjab,Chandigarh,BSc Data Science,2,Hostel,simran.kaur@example.com,9876501026
STU027,Abhishek Das,19,Male,Assam,Guwahati,BBA,1,Hostel,abhishek.das@example.com,9876501027
STU028,Lakshmi Rao,21,Female,Andhra Pradesh,Vijayawada,BCom,3,Day Scholar,lakshmi.rao@example.com,9876501028
STU029,Yash Thakur,20,Male,Himachal Pradesh,Shimla,BTech,2,Hostel,yash.thakur@example.com,9876501029
STU030,Farah Khan,22,Female,Jammu and Kashmir,Srinagar,MBA,1,Hostel,farah.khan@example.com,9876501030
STU031,Ananya Iyer,20,Female,Karnataka,Mysuru,BBA,2,Hostel,ananya.iyer2@example.com,9876501031
STU032,Suresh Naidu,21,Male,Andhra Pradesh,Visakhapatnam,BSc Data Science,3,Day Scholar,suresh.naidu@example.com,9876501005
STU033,Nandini Roy,19,Female,Chhattisgarh,Raipur,BCom,1,Hostel,priya.das@example.com,9876501033
STU034,Aman Tripathi,22,Male,Uttar Pradesh,Varanasi,MBA,1,Day Scholar,aman.tripathi@example.com,9876501034
STU034,Aman Tripathi,22,Male,Uttar Pradesh,Prayagraj,MBA,1,Hostel,aman.tripathi2@example.com,9876501035
STU006,Sneha Patil,21,Female,Maharashtra,Nashik,BBA,3,Hostel,sneha.patil@example.com,9876501006
STU012,Aditi Joshi,20,Female,Gujarat,Ahmedabad,BTech,2,Day Scholar,aditi.joshi@example.com,9876501012
STU018,Divya Shetty,19,Female,Karnataka,Mangaluru,BCom,1,Hostel,divya.shetty@example.com,9876501018
STU024,Ritika Chawla,21,Female,Punjab,Amritsar,BTech,3,Hostel,ritika.chawla@example.com,9876501024
STU029,Yash Thakur,20,Male,Himachal Pradesh,Shimla,BTech,2,Hostel,yash.thakur@example.com,9876501029"""
    return pd.read_csv(io.StringIO(csv_text))


def rebuild_demo():
    """Rebuild a dataset statistically equivalent to demo.csv (263 rows).

    Same columns, same types, same ranges and category mix, so every cell
    below still runs if the original file is missing."""
    rng = np.random.default_rng(42)
    n = 263
    frame = pd.DataFrame({
        'age': rng.normal(38.69, 13.45, n).clip(17, 90).round().astype(int),
        'workclass': rng.choice([' Private', ' Self-emp-not-inc', ' Local-gov', ' ?', ' State-gov', ' Federal-gov', ' Self-emp-inc'], n, p=[0.6844, 0.0722, 0.0684, 0.0532, 0.0494, 0.0418, 0.0306]),
        'fnlwgt': rng.normal(1.915e+05, 1.09e+05, n).clip(24215, 635913).round().astype(int),
        'eductation': rng.choice(['HS-grad', 'Some-college', 'Bachelors', 'Masters', '11th', 'Assoc-acdm', 'Assoc-voc', '7th-8th', '9th', 'Doctorate', '10th', 'Prof-school', '5th-6th', '1st-4th', 'Preschool'], n, p=[0.3118, 0.2167, 0.1635, 0.0608, 0.0456, 0.0456, 0.0418, 0.0228, 0.019, 0.019, 0.019, 0.0152, 0.0076, 0.0076, 0.004]),
        'education-num': rng.normal(10.15, 2.689, n).clip(1, 16).round().astype(int),
        'marital_status': rng.choice(['Married-civ-spouse', 'Never-married', 'Divorced', 'Separated', 'Widowed', 'Married-spouse-absent', 'Married-AF-spouse'], n, p=[0.4715, 0.3118, 0.1521, 0.0266, 0.0228, 0.0114, 0.0038]),
        'occupation': rng.choice(['Sales', 'Prof-specialty', 'Craft-repair', 'Exec-managerial', 'Other-service', 'Adm-clerical', 'Machine-op-inspct', '?', 'Handlers-cleaners', 'Tech-support', 'Transport-moving', 'Protective-serv', 'Farming-fishing'], n, p=[0.1331, 0.1293, 0.1217, 0.1179, 0.1179, 0.1103, 0.057, 0.0532, 0.0418, 0.0418, 0.0342, 0.0228, 0.019]),
        'relationship': rng.choice(['Husband', 'Not-in-family', 'Own-child', 'Unmarried', 'Wife', 'Other-relative'], n, p=[0.3992, 0.2738, 0.1521, 0.0837, 0.0608, 0.0304]),
        'race': rng.choice(['White', 'Black', 'Asian-Pac-Islander', 'Amer-Indian-Eskimo', 'Other'], n, p=[0.8061, 0.1369, 0.038, 0.0114, 0.0076]),
        'gender': rng.choice(['Male', 'Female'], n, p=[0.6768, 0.3232]),
        'capitol_gain': rng.normal(597.9, 3028, n).clip(0, 34095).round().astype(int),
        'capitol_loss': rng.normal(124.9, 465.7, n).clip(0, 2206).round().astype(int),
        'hrs_per_week': rng.normal(40.08, 11.28, n).clip(1, 80).round().astype(int),
        'native-country': rng.choice([' United-States', ' Mexico', ' ?', ' Cuba', ' Puerto-Rico', ' England', ' Iran', ' Jamaica', ' India', ' South', ' Honduras', ' Canada', ' Germany', ' Philippines', ' Italy', ' Poland', ' Columbia', ' Cambodia'], n, p=[0.8707, 0.0304, 0.0266, 0.0076, 0.0076, 0.0076, 0.0076, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0038, 0.0039]),
        'classifiaction_va': rng.choice([' <=50K', ' >50K'], n, p=[0.7681, 0.2319]),
    })
    return frame

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


# 01 · Duplicate Detection & Resolution
### Why "duplicate" is four different questions, not one

**Dataset:** `data/student_data.csv` — 40 student records from an Indian PGDM
programme roster: ID, name, age, gender, state, city, programme, year,
residence type, email, phone.

**Why this dataset is a good teacher:** it was built with **four distinct
kinds of duplication** baked in on purpose, and they need four different
techniques to tell apart:

1. **True duplicates** — the exact same record, copy-pasted.
2. **Same name, different person** — a coincidence, not a duplicate.
3. **Same ID, conflicting data** — a primary-key collision, not a name issue
   at all.
4. **Same "unique" value on an unrelated record** — an email or phone number
   reused across two different people.

Only #1 should ever be deleted automatically. The other three need to be
*found*, but resolving them takes human judgment a script shouldn't make for
you.

**Learning objectives**
1. Audit for duplication along **every column that ought to be unique**, not
   just full rows.
2. Reproduce the original notebook's technique on real data and watch it
   **delete two students who were never duplicates at all**.
3. Build one small, reusable uniqueness-audit function instead of one-off
   checks per column.
4. Compare `keep='first'` / `'last'` / `False` on the *right* subset, with
   the row-count consequences made explicit.

**Contents**
1. Load & first look
2. Audit — four checks for four kinds of duplication
3. Feature split — `Student_Name` → `First_Name` + `Last_Name`
4. The original technique, reproduced — and why it's dangerous
5. Doing it correctly — true full-row duplicates only
6. Collisions a full-row check can't catch
7. Resolving a same-ID conflict
8. Method showcase — `keep='first'` vs `'last'` vs `False`
9. Final cleaned dataset & data-quality scorecard

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 150)

DATA_PATH = dataset_path('student_data.csv', rebuild=rebuild_student_data)   # resolved wherever the datasets folder lives

## 1 · Load & first look

In [3]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(8)

Shape: 40 rows x 11 columns


,Student_ID,Student_Name,Age,Gender,State,City,Program,Year_of_Study,Residence_Type,Email,Phone_Number
0,STU001,Aarav Sharma,20,Male,Uttar Pradesh,Lucknow,BBA,2,Hostel,aarav.sharma@example.com,9876501001
1,STU002,Ananya Iyer,21,Female,Tamil Nadu,Chennai,BCom,3,Day Scholar,ananya.iyer@example.com,9876501002
2,STU003,Rohan Desai,19,Male,Maharashtra,Pune,BSc Data Science,1,Hostel,rohan.desai@example.com,9876501003
3,STU004,Meera Nair,22,Female,Kerala,Kochi,MBA,1,Hostel,meera.nair@example.com,9876501004
4,STU005,Arjun Reddy,20,Male,Telangana,Hyderabad,BTech,2,Day Scholar,arjun.reddy@example.com,9876501005
5,STU006,Sneha Patil,21,Female,Maharashtra,Nashik,BBA,3,Hostel,sneha.patil@example.com,9876501006
6,STU007,Vikram Singh,23,Male,Rajasthan,Jaipur,MBA,2,Hostel,vikram.singh@example.com,9876501007
7,STU008,Priya Das,19,Female,West Bengal,Kolkata,BCom,1,Day Scholar,priya.das@example.com,9876501008


## 2 · Audit — four checks for four kinds of duplication

The original notebook only ever asked one question ("is this whole row, or
this one column, repeated?"). We ask it of every column that's *supposed*
to be a unique identifier — because a repeated `Student_ID` and a repeated
`Email` are both data-integrity problems, even when no single row is a
full duplicate of another.

In [4]:
# --- Check A: exact full-row duplicates ------------------------------------
exact_dupes = df.duplicated(keep=False)
print(f"Rows involved in an exact full-row duplicate: {exact_dupes.sum()}")

Rows involved in an exact full-row duplicate: 10


In [5]:
# --- Check B: Student_ID should be a primary key — is it? ------------------
id_dupes = df.duplicated(subset="Student_ID", keep=False)
print(f"Rows sharing a Student_ID with another row: {id_dupes.sum()}")

Rows sharing a Student_ID with another row: 12


In [6]:
# --- Check C: Email should be one-per-person — is it? -----------------------
email_dupes = df.duplicated(subset="Email", keep=False)
print(f"Rows sharing an Email with another row: {email_dupes.sum()}")

Rows sharing an Email with another row: 12


In [7]:
# --- Check D: Phone_Number should be one-per-person — is it? ---------------
phone_dupes = df.duplicated(subset="Phone_Number", keep=False)
print(f"Rows sharing a Phone_Number with another row: {phone_dupes.sum()}")

Rows sharing a Phone_Number with another row: 12


Four checks, four different counts. That gap between them *is* the finding:
some rows will show up in more than one check (a true duplicate shares
everything, so it trips all four), but others will show up in exactly one —
and those are the ones a naive "drop the duplicates" pass would miss
entirely, or delete for the wrong reason. Sections 4–7 pull them apart.

## 3 · Feature split — `Student_Name` → `First_Name` + `Last_Name`

The original notebook's technique operated on a `"First Name"` column that
assumed the roster already had first and last names in separate fields.
This roster doesn't — `Student_Name` holds both together (`"Aarav
Sharma"`). Splitting it out is what makes the original technique possible
to even run here.

In [8]:
df[["First_Name", "Last_Name"]] = df["Student_Name"].str.split(" ", n=1, expand=True)

# Confirm the split is clean before relying on it — every name should have
# produced exactly two non-null parts.
incomplete_splits = df["Last_Name"].isna().sum()
print(f"Names that didn't split into exactly two parts: {incomplete_splits}")
df[["Student_Name", "First_Name", "Last_Name"]].head()

Names that didn't split into exactly two parts: 0


,Student_Name,First_Name,Last_Name
0,Aarav Sharma,Aarav,Sharma
1,Ananya Iyer,Ananya,Iyer
2,Rohan Desai,Rohan,Desai
3,Meera Nair,Meera,Nair
4,Arjun Reddy,Arjun,Reddy


## 4 · The original technique, reproduced — and why it's dangerous

The original notebook's exact approach:

```python
df.sort_values("First Name", inplace=True)
df.drop_duplicates(subset="First Name", keep=False, inplace=True)
```

i.e. *any* row whose first name appears more than once gets deleted —
**both/all copies**, no exceptions. We run that exact logic here, then check
what it actually removed.

In [9]:
by_first_name = df.sort_values("First_Name")
rows_deleted_by_original = by_first_name[
    by_first_name.duplicated(subset="First_Name", keep=False)
]
print(f"Rows the original technique would delete: {len(rows_deleted_by_original)}")
rows_deleted_by_original[["Student_ID", "First_Name", "Last_Name", "City"]]

Rows the original technique would delete: 16


,Student_ID,First_Name,Last_Name,City
0,STU001,Aarav,Sharma,Lucknow
20,STU021,Aarav,Sharma,Ludhiana
36,STU012,Aditi,Joshi,Ahmedabad
11,STU012,Aditi,Joshi,Ahmedabad
34,STU034,Aman,Tripathi,Prayagraj
33,STU034,Aman,Tripathi,Varanasi
1,STU002,Ananya,Iyer,Chennai
30,STU031,Ananya,Iyer,Mysuru
17,STU018,Divya,Shetty,Mangaluru
37,STU018,Divya,Shetty,Mangaluru


In [10]:
# Cross-check each deleted row against the TRUE full-row-duplicate flag from
# Section 2 — a row deleted here that ISN'T a true duplicate is a false
# positive: a real, distinct student erased for sharing a first name.
false_positives = rows_deleted_by_original[~exact_dupes.reindex(rows_deleted_by_original.index)]
print(f"Of those, rows that are NOT true duplicates (i.e. wrongly deleted): {len(false_positives)}")
false_positives[["Student_ID", "First_Name", "Last_Name", "Age", "City", "Email"]]

Of those, rows that are NOT true duplicates (i.e. wrongly deleted): 6


,Student_ID,First_Name,Last_Name,Age,City,Email
0,STU001,Aarav,Sharma,20,Lucknow,aarav.sharma@example.com
20,STU021,Aarav,Sharma,22,Ludhiana,aarav.sharma2@example.com
34,STU034,Aman,Tripathi,22,Prayagraj,aman.tripathi2@example.com
33,STU034,Aman,Tripathi,22,Varanasi,aman.tripathi@example.com
1,STU002,Ananya,Iyer,21,Chennai,ananya.iyer@example.com
30,STU031,Ananya,Iyer,20,Mysuru,ananya.iyer2@example.com


**Aarav Sharma (`STU001`, Lucknow) and Aarav Sharma (`STU021`, Ludhiana) are
two different students** — different ages, different cities, different
emails. Same for the two Ananya Iyers: four rows, two pairs of genuinely
different people, deleted purely because a first name matched.

The other two rows in that list are `STU034`'s pair — and they belong here
for a *different* reason. Those two rows aren't true duplicates either (city,
residence type, email, and phone all differ), so the technique didn't
correctly identify a duplicate there — it got the right row count out of
Section 6's real problem (a same-ID conflict) **by accident**, because the
name happened to match too. Change the last name on either row and this
technique would miss it entirely.

Six rows deleted, zero of them a genuine duplicate: four are collateral
damage, two are a real problem "solved" for the wrong reason. A name is not
an identifier; treating it like one is unreliable in both directions.

## 5 · Doing it correctly — true full-row duplicates only

The fix is narrow: only delete a row when **every column** matches another
row, not just one. That's `duplicated()` / `drop_duplicates()` with no
`subset` argument at all.

In [11]:
true_dupe_pairs = df[df.duplicated(keep=False)].sort_values("Student_ID")
true_dupe_pairs[["Student_ID", "Student_Name", "City", "Email"]]

,Student_ID,Student_Name,City,Email
5,STU006,Sneha Patil,Nashik,sneha.patil@example.com
35,STU006,Sneha Patil,Nashik,sneha.patil@example.com
11,STU012,Aditi Joshi,Ahmedabad,aditi.joshi@example.com
36,STU012,Aditi Joshi,Ahmedabad,aditi.joshi@example.com
17,STU018,Divya Shetty,Mangaluru,divya.shetty@example.com
37,STU018,Divya Shetty,Mangaluru,divya.shetty@example.com
23,STU024,Ritika Chawla,Amritsar,ritika.chawla@example.com
38,STU024,Ritika Chawla,Amritsar,ritika.chawla@example.com
28,STU029,Yash Thakur,Shimla,yash.thakur@example.com
39,STU029,Yash Thakur,Shimla,yash.thakur@example.com


In [12]:
before = df.shape[0]
df_deduped = df.drop_duplicates(keep="first")
print(f"Rows before: {before}   Rows after removing TRUE duplicates only: {df_deduped.shape[0]}")
print(f"({before - df_deduped.shape[0]} exact duplicate rows removed — the 4 false-positive")
print(" students from Section 4 are correctly kept.)")

Rows before: 40   Rows after removing TRUE duplicates only: 35
(5 exact duplicate rows removed — the 4 false-positive
 students from Section 4 are correctly kept.)


## 6 · Collisions a full-row check can't catch

`STU034` shares its Student_ID with nothing on a full-row check (its two
rows *aren't* identical — Section 2's Check A wouldn't flag it) and shares
nothing on a first-name check applied honestly at full-name grain either
— it only surfaces via **Check B** (ID) directly. Likewise, the email and
phone collisions from Checks C/D involve entirely different names and IDs —
no name-based or full-row check could ever find them. This is exactly why
Section 2 ran four separate checks instead of one.

In [13]:
def uniqueness_report(frame: pd.DataFrame, id_like_columns: list[str]) -> pd.DataFrame:
    '''For each column that should be unique per person, report how many
    rows collide on it and whether those colliding rows are otherwise
    identical (a true duplicate) or genuinely conflicting (needs review).'''
    rows = []
    for col in id_like_columns:
        collided = frame[frame.duplicated(subset=col, keep=False)]
        if collided.empty:
            continue
        for value, group in collided.groupby(col):
            is_true_dupe = group.duplicated(keep=False).all()
            rows.append({
                "column": col, "value": value, "n_rows": len(group),
                "status": "true duplicate" if is_true_dupe else "CONFLICTING DATA — needs review",
            })
    return pd.DataFrame(rows)


report = uniqueness_report(df, ["Student_ID", "Email", "Phone_Number"])
report

,column,value,n_rows,status
0,Student_ID,STU006,2,true duplicate
1,Student_ID,STU012,2,true duplicate
2,Student_ID,STU018,2,true duplicate
3,Student_ID,STU024,2,true duplicate
4,Student_ID,STU029,2,true duplicate
5,Student_ID,STU034,2,CONFLICTING DATA — needs review
6,Email,aditi.joshi@example.com,2,true duplicate
7,Email,divya.shetty@example.com,2,true duplicate
8,Email,priya.das@example.com,2,CONFLICTING DATA — needs review
9,Email,ritika.chawla@example.com,2,true duplicate


Two rows the report calls out as genuinely conflicting, beyond the `STU034`
ID collision:

- `priya.das@example.com` belongs to both `STU008` (Priya Das) and `STU033`
  (Nandini Roy) — two different names sharing one email.
- `9876501005` belongs to both `STU005` (Arjun Reddy) and `STU032` (Suresh
  Naidu) — two different names sharing one phone number.

Neither looks like a data-entry accident the way `STU034` does (there, at
least the name matches); these look more like placeholder/synthetic values
that happened to collide. Either way, the right move is the same: **surface
it, don't silently pick a winner.**

## 7 · Resolving a same-ID conflict

`STU034` is the one case where the *same identity* clearly has two
conflicting records — worth looking at directly rather than only through
the summary report above.

In [14]:
df[df["Student_ID"] == "STU034"][
    ["Student_ID", "Student_Name", "City", "Residence_Type", "Email", "Phone_Number"]
]

,Student_ID,Student_Name,City,Residence_Type,Email,Phone_Number
33,STU034,Aman Tripathi,Varanasi,Day Scholar,aman.tripathi@example.com,9876501034
34,STU034,Aman Tripathi,Prayagraj,Hostel,aman.tripathi2@example.com,9876501035


Same ID, same name, same age — but a different city, residence type, email,
and phone. That pattern (everything demographic matches, everything
*contact-related* differs) looks like a re-registration or a corrected
resubmission rather than two unrelated people — but that's a judgment call
for someone with access to the source system, not something to guess at in
a notebook. We flag it rather than resolve it.

In [15]:
flagged_for_review = pd.concat([
    df[df["Student_ID"] == "STU034"],
    df[df["Email"] == "priya.das@example.com"],
    df[df["Phone_Number"] == "9876501005"],
]).drop_duplicates()

print(f"{len(flagged_for_review)} rows flagged for manual review (not deleted, not auto-resolved).")
flagged_for_review[["Student_ID", "Student_Name", "City", "Email", "Phone_Number"]]

4 rows flagged for manual review (not deleted, not auto-resolved).


,Student_ID,Student_Name,City,Email,Phone_Number
33,STU034,Aman Tripathi,Varanasi,aman.tripathi@example.com,9876501034
34,STU034,Aman Tripathi,Prayagraj,aman.tripathi2@example.com,9876501035
7,STU008,Priya Das,Kolkata,priya.das@example.com,9876501008
32,STU033,Nandini Roy,Raipur,priya.das@example.com,9876501033


## 8 · Method showcase — `keep='first'` vs `'last'` vs `False`

All three operate on the *correct* subset (a full-row match) — the
difference is only which copy (if any) survives.

In [16]:
variants = {
    "keep='first' (keep the earliest copy)": df.drop_duplicates(keep="first"),
    "keep='last'  (keep the latest copy)":   df.drop_duplicates(keep="last"),
    "keep=False   (discard every copy)":     df.drop_duplicates(keep=False),
}

pd.DataFrame({
    "resulting_row_count": {name: len(result) for name, result in variants.items()},
}).rename_axis("strategy")

,resulting_row_count
strategy,
keep='first' (keep the earliest copy),35
keep='last' (keep the latest copy),35
keep=False (discard every copy),30


| Strategy | Rows kept from a duplicate pair | When to reach for it |
|---|---|---|
| `keep='first'` | The first occurrence | Default choice — you trust earlier entries (e.g. original submission over a resend) |
| `keep='last'` | The last occurrence | You trust later entries (e.g. a corrected re-submission should win) |
| `keep=False` | Neither | You don't trust *either* copy enough to pick one, and would rather lose the row than guess |

For this roster, `keep='first'` is the right default — nothing suggests the
second copy of any true duplicate is more trustworthy than the first.

## 9 · Final cleaned dataset & data-quality scorecard

In [17]:
final_df = df.drop_duplicates(keep="first").drop(columns=["First_Name", "Last_Name"])

print("Data-quality scorecard")
print("=" * 46)
print(f"{'Starting rows':38s}: {len(df)}")
print(f"{'Exact duplicate rows removed':38s}: {len(df) - len(df.drop_duplicates(keep='first'))}")
print(f"{'Rows flagged for manual review':38s}: {len(flagged_for_review)}")
print(f"{'Final row count':38s}: {len(final_df)}")
print(f"{'Distinct Student_IDs in final data':38s}: {final_df['Student_ID'].nunique()}")

Data-quality scorecard
Starting rows                         : 40


Exact duplicate rows removed          : 5
Rows flagged for manual review        : 4
Final row count                       : 35
Distinct Student_IDs in final data    : 34


That one-row gap between `35` final rows and `34` distinct IDs is exactly
`STU034`, still sitting in the data twice — on purpose. It wasn't deleted,
because deleting it would mean guessing which of the two conflicting records
is correct. It's flagged in `flagged_for_review` for whoever owns the source
system to actually resolve.

### Recap

- **Four checks, not one** — full-row, `Student_ID`, `Email`, `Phone_Number`
  — because each catches a different failure mode.
- **The original name-based technique, run on real data, deleted two
  genuine students.** Reproducing it wasn't academic — it's the clearest
  possible demonstration of why a name is not an identifier.
- **A same-ID conflict (`STU034`) and two cross-identity value collisions**
  were found precisely because Section 2 checked columns individually
  instead of only checking whole rows.
- **Flag, don't guess** — every ambiguous case went into a review list
  instead of being silently resolved one way or the other.

**Where this feeds next:** a cleaned, de-duplicated table like `final_df`
is the right starting point *before* any train/validation/test split —
duplicate rows leaking across a split would let a model "cheat" by seeing
the same record in both training and evaluation. That's exactly where
`02_train_validation_test_splitting.ipynb` picks up.

### ⚡ Beyond the syllabus — the order of operations, and what each wrong order costs you

Cleaning steps are not interchangeable. Get the sequence wrong and you leak information into your test set without any error appearing. Here is the correct order, with a demonstration of exactly how much a leak inflates your score — which is the number that makes the point.

In [18]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

print("""
THE CORRECT ORDER
  1. Load the raw data
  2. Remove duplicates                 <- must be before the split
  3. Fix obvious errors / impossible values
  4. SPLIT into train / validation / test
  5. Fit imputers, scalers and encoders on TRAIN ONLY
  6. Apply those fitted objects to validation and test
  7. Train, tune on validation, and touch the test set exactly once

WHAT EACH WRONG ORDER COSTS YOU
  De-duplicating after the split   -> the same row sits in train and test. Test score
                                      measures memorisation. Inflation: large.
  Scaling before the split         -> the mean and std were computed using test rows,
                                      so the test set influenced the transformation.
  Imputing before the split        -> the median used to fill training gaps was computed
                                      partly from test data.
  Selecting features on all data   -> you chose the features using the answers you were
                                      about to be tested on. The worst one of the four.
""")

# ---- Measuring the duplicate leak ------------------------------------------
rng = np.random.default_rng(7)
n = 600
base = pd.DataFrame({
    'x1': rng.normal(0, 1, n),
    'x2': rng.normal(0, 1, n),
    'x3': rng.normal(0, 1, n),
})
base['y'] = ((base['x1'] + base['x2'] * .5 + rng.normal(0, .9, n)) > 0).astype(int)

# 30% of rows duplicated, exactly as a repeated import would produce
duplicated = pd.concat([base, base.sample(frac=.3, random_state=1)], ignore_index=True)

model = make_pipeline(StandardScaler(), LogisticRegression())

# WRONG: split first, so copies of the same row land on both sides
tr_w, te_w = train_test_split(duplicated, test_size=.3, random_state=0, stratify=duplicated['y'])
leaky = model.fit(tr_w[['x1','x2','x3']], tr_w['y']).score(te_w[['x1','x2','x3']], te_w['y'])

# RIGHT: de-duplicate first, then split
clean = duplicated.drop_duplicates()
tr_r, te_r = train_test_split(clean, test_size=.3, random_state=0, stratify=clean['y'])
honest = model.fit(tr_r[['x1','x2','x3']], tr_r['y']).score(te_r[['x1','x2','x3']], te_r['y'])

overlap = pd.merge(tr_w, te_w, on=['x1','x2','x3','y']).shape[0]

print(f"Rows before de-duplication : {len(duplicated)}")
print(f"Rows after                 : {len(clean)}")
print(f"Rows appearing in BOTH train and test (wrong order): {overlap}\n")
print(f"Test accuracy, split first (leaky)   : {leaky:.4f}")
print(f"Test accuracy, de-duplicate first    : {honest:.4f}")
print(f"Inflation from the leak              : {(leaky - honest)*100:+.2f} percentage points")
print("\nThe leak makes the model look better than it is. Nothing errors, nothing warns —")
print("which is exactly why the ordering has to be a habit rather than something you check.")


THE CORRECT ORDER
  1. Load the raw data
  2. Remove duplicates                 <- must be before the split
  3. Fix obvious errors / impossible values
  4. SPLIT into train / validation / test
  5. Fit imputers, scalers and encoders on TRAIN ONLY
  6. Apply those fitted objects to validation and test
  7. Train, tune on validation, and touch the test set exactly once

WHAT EACH WRONG ORDER COSTS YOU
  De-duplicating after the split   -> the same row sits in train and test. Test score
                                      measures memorisation. Inflation: large.
  Scaling before the split         -> the mean and std were computed using test rows,
                                      so the test set influenced the transformation.
  Imputing before the split        -> the median used to fill training gaps was computed
                                      partly from test data.
  Selecting features on all data   -> you chose the features using the answers you were
                     

---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Count duplicates | `df.duplicated().sum()` |
| On chosen columns | `df.duplicated(subset=[…]).sum()` |
| Show every copy | `df[df.duplicated(keep=False)]` |
| Remove | `df.drop_duplicates()` |
| Keep the most recent | `df.sort_values('date').drop_duplicates(subset='id', keep='last')` |
| Resolve by aggregating | `df.groupby('id').agg({'score':'mean'})` |
| Renumber afterwards | `.reset_index(drop=True)` |
| Check no overlap | `pd.merge(train, test, on=cols).shape[0] == 0` |

### Adapting this in the exam

- 'Prepare the data for modelling' → de-duplicate, fix errors, split, then fit transformers on train only.
- 'Why is my test accuracy suspiciously high?' → check for duplicates across the split, and for scaling or imputing done before it.

### Traps that cost marks

- De-duplicating **after** the split leaves copies on both sides and inflates your test score.
- When duplicate rows disagree on a value, you must decide which to keep — most recent, highest quality, or an average. Say which and why.
- An ID column defeats duplicate detection. Exclude it via `subset=`.
- `drop_duplicates()` keeps the first occurrence. If later rows are the corrections, use `keep='last'`.
- Always report the row count before and after — that's the evidence the step happened.